# Hyperparameter Search with Optuna

**Notebook 3 of 3.** A grid search tries every combination you listed and learns nothing on
the way. With five settings to choose, listing enough values is not possible — three values
each would already be 243 fits, and three values is not a search. This notebook uses a
**Bayesian** search instead: one that builds a model of which settings have been working and
spends its next trial where that model says to look.

## 1. The data, and the bar

`data/california_housing.csv`: 20,640 districts, the target `MedHouseVal` in units of
**$100,000**, so error is RMSE in those units. The same table the previous notebook used.

Untouched `XGBRegressor` defaults are what the search has to beat.

In [ ]:
import time
import warnings
from pathlib import Path

# optuna imports tqdm, which warns about ipywidgets on a plain kernel. It fires on
# import, so the filter has to be set before optuna is imported, not after
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor
from xgboost.callback import TrainingCallback

RSEED = 42
N_JOBS = 4          # see notebook 02: real cores, not threads. Raise it to suit your machine
ROUNDS = 400        # an upper bound on trees; the search chooses the learning rate to match

# Optuna logs a line per trial by default, which is 40 lines of noise here
optuna.logging.set_verbosity(optuna.logging.WARNING)


def rmse(truth, prediction):
    """RMSE in units of $100,000."""
    return root_mean_squared_error(truth, prediction)

In [ ]:
df = pd.read_csv("data/california_housing.csv")

y = df["MedHouseVal"]
X = pd.get_dummies(df.drop(columns="MedHouseVal"), columns=["location"]).astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RSEED
)

# the search needs somewhere to score its trials, and it must not be the test districts.
# This slice comes out of the TRAINING rows, so the test set stays untouched until the end
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RSEED
)

defaults = XGBRegressor(random_state=RSEED, n_jobs=N_JOBS)
defaults.fit(X_fit, y_fit)

print(f"{len(X_fit)} districts to fit, {len(X_val)} to score trials on, {len(X_test)} held back")
print(f"defaults: {rmse(y_val, defaults.predict(X_val)):.4f} on the validation districts")

## 2. The five settings

XGBoost exposes dozens. These five are the ones worth a search on a table like this, and
each one controls a different way of spending or saving complexity:

- **`learning_rate`** scales down every tree's contribution. Searched on a **log** scale,
  from 0.01 to 0.3, because the distance from 0.01 to 0.02 matters as much as from 0.1 to
  0.2 — a linear range would spend almost all its trials above 0.1.
- **`max_depth`**, 3 to 10, caps how deep any one tree goes.
- **`subsample`**, 0.5 to 1.0, fits each tree on a fraction of the rows.
- **`colsample_bytree`**, 0.5 to 1.0, gives each tree a fraction of the columns.
- **`reg_lambda`**, 0.01 to 100 on a log scale, penalizes large leaf values as ridge
  regression does.

`n_estimators` is **not** searched. It is fixed at 400 and the search picks a
`learning_rate` to suit, which is the same two-settings-in-one relationship boosting always
has — searching both would spend trials rediscovering it.

## 3. The objective function

Optuna is **define-by-run**: there is no search space object to declare up front. You write
an ordinary function, and wherever it needs a number it asks the `trial` for one. The trial
decides what to hand back, and the function returns the score Optuna should minimize — here
the RMSE on the validation districts, which is what the search will drive down.

In [ ]:
def suggest_settings(trial):
    """Ask the trial for one value of each of the five settings.

    Args:
        trial: the Optuna trial being run. log=True searches the ratio rather than the
            difference, which is what a rate or a penalty strength needs.

    Returns:
        dict: XGBRegressor keyword arguments.
    """
    return dict(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 100.0, log=True),
    )


def objective(trial):
    """Fit one XGBRegressor and report its RMSE on the validation districts."""
    model = XGBRegressor(n_estimators=ROUNDS, random_state=RSEED, n_jobs=N_JOBS,
                         **suggest_settings(trial))
    model.fit(X_fit, y_fit)
    return rmse(y_val, model.predict(X_val))

## 4. The study

A **study** runs the objective many times and keeps the results. The sampler is what makes
it a Bayesian search: Optuna's default **TPE** — Tree-structured Parzen Estimator — models
which settings produced good scores and which produced bad ones, and draws each new trial
from where that model thinks good scores live. Early trials are near-random, because it has
nothing to learn from yet; later ones are not.

> **Exercise 1.** Create a study that **minimizes**, with `TPESampler(seed=RSEED)` so the
> result reproduces, run `objective` for 40 trials, and print the best value and the best
> parameters. Then plot each trial's score against its trial number, with the running best
> on top. Allow about two and a half minutes.

<details><summary>
Click here for a hint...
</summary>

`optuna.create_study(direction=..., sampler=...)` then `study.optimize(objective,
n_trials=40)`. Every trial's score is in `[t.value for t in study.trials]`, and
`np.minimum.accumulate` turns that into the running best.

</details>

In [ ]:
# your code: a study minimizing `objective` over 40 TPE trials, its best value and
# parameters printed, and a plot of trial score against trial number

<details><summary>
Click here to compare your answer...
</summary>

**Best validation RMSE 0.4587**, against the defaults' 0.4934.

| setting | chosen |
| :-- | --: |
| `learning_rate` | 0.0402 |
| `max_depth` | 9 |
| `subsample` | 0.8536 |
| `colsample_bytree` | 0.5585 |
| `reg_lambda` | 0.0711 |

The printed lines are the search working. **The spread of the trial scores falls from
0.024 to 0.004** between the first ten trials and the last ten, and the worst trial improves
from 0.5599 to 0.4754. TPE stopped proposing settings from regions that had already failed —
that is the whole difference between a Bayesian search and drawing at random, and it is
visible as the grey dots tightening toward the right of the plot.

The red line says something useful about **how long to run**. Only **47% of the total gain
had arrived by trial 12**, and 63% by trial 20; the last twenty trials produced the rest.
A short study is far better than none, and here a longer one kept paying.

</details>

## 5. Pruning: stopping the hopeless trials

Section 4 fitted 400 trees forty times, including for settings that were clearly going
nowhere after thirty. **Pruning** cuts those off. The trial reports its score as it goes,
Optuna compares that against the other trials at the same point, and a trial doing worse
than the median is stopped where it stands.

Boosting is the natural fit, because a boosted model produces a score after every tree.
Below, a `TrainingCallback` hands XGBoost's validation RMSE to the trial once per round.

In [ ]:
class ReportEachRound(TrainingCallback):
    """Report the validation RMSE to Optuna after every boosting round.

    XGBoost calls after_iteration once per tree. Reporting there is what gives the
    pruner something to judge: a trial that is behind after 50 trees rarely catches up.
    """

    def __init__(self, trial):
        self.trial = trial

    def after_iteration(self, model, epoch, evals_log):
        """Return False to let training continue; raise TrialPruned to stop it."""
        self.trial.report(evals_log["validation_0"]["rmse"][-1], epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()
        return False


def pruning_objective(trial):
    """The same five settings, with the trial reporting after every tree."""
    model = XGBRegressor(n_estimators=ROUNDS, random_state=RSEED, n_jobs=N_JOBS,
                         eval_metric="rmse", callbacks=[ReportEachRound(trial)],
                         **suggest_settings(trial))
    # eval_set is what the callback reads; verbose=False silences one line per tree
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
    return rmse(y_val, model.predict(X_val))

In [ ]:
start = time.perf_counter()

pruned_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RSEED),
    # no pruning for the first 5 trials, since there is nothing to compare against,
    # and none inside the first 50 rounds of any trial, which are noise
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=50),
)
pruned_study.optimize(pruning_objective, n_trials=40)

states = pd.Series([t.state.name for t in pruned_study.trials]).value_counts()
print(f"best validation RMSE {pruned_study.best_value:.4f}   "
      f"[{time.perf_counter() - start:.0f}s; compare the unpruned study above]")
print(f"\n{states.to_string()}")

**19 of the 40 trials were stopped early**, and the study finished in roughly 40% less
time than the unpruned one above. The exact seconds depend on the machine; the count of
pruned trials does not.

It is not free: the best validation score came out **0.4618** rather than 0.4587, because a
trial that would have won was killed before it recovered. Pruning trades a small risk of
that against a large saving, and on a search long enough to be worth running, the saving is
what lets you run more trials.

## 6. Which of the five actually mattered

A study records every trial, so it can be asked which settings the scores were sensitive
to. This is worth reading before the next search: a setting that explains nothing can be
dropped from the space, and the trials spent on it given to one that matters.

In [ ]:
importance = optuna.importance.get_param_importances(pruned_study)
for name, share in importance.items():
    print(f"  {name:<18} {share:.3f}  {'=' * int(share * 50)}")

# which trials was that computed over? Only the ones that finished -- and section 5's
# pruner is what decided which those were
by_state = pd.DataFrame(
    [(t.state.name, t.params["learning_rate"]) for t in pruned_study.trials],
    columns=["state", "learning_rate"],
)
print("\nlearning_rate, among the trials that finished and among those pruned:")
print(by_state.groupby("state")["learning_rate"].agg(["count", "min", "max"]).to_string())

**`colsample_bytree` 0.48 and `max_depth` 0.41 account for nearly nine tenths of it**, and
`learning_rate` comes last at 0.02. Taken at face value that says the learning rate hardly
matters in a boosted model, which is not true — and the second table says why.

Optuna computes importances over the trials that **finished**. Section 5's pruner decided
which those were, and it did not choose evenly: the surviving trials' learning rates span
**0.011 to 0.144**, while the pruned ones reach **0.255**. A high rate converges quickly and
then overfits, so it slips behind the median around round 50 and is stopped. The trials that
would have shown the rate mattering are precisely the ones missing from the calculation.

**An importance is a statement about the trials you kept.** Run
`optuna.importance.get_param_importances()` on your section 4 study, which completed all
forty, and the ranking comes out different again. Read these numbers as a description of one
search, never as a property of the algorithm.

## 7. One number, at the end

Everything so far was chosen on the validation districts, so the best validation score is
**not** an estimate of how this model handles new data — the search looked at that slice
forty times and found what suits it.

Two steps close the gap. Cross-validate the chosen settings on the **training** rows, which
re-uses no slice the search optimized against, and only then fit on all the training
districts and score the test set once.

> **Exercise 2.** Take `pruned_study.best_params`, cross-validate them against untouched
> defaults with 3-fold `KFold` on the training rows, then fit both on all of `X_train` and
> score each once on the test districts. Does the gain survive?

<details><summary>
Click here for a hint...
</summary>

`cross_val_score(..., scoring="neg_root_mean_squared_error")` returns negated values, so
negate them back. Keep `n_estimators=ROUNDS` on the tuned model, since that is what the
search assumed.

</details>

In [ ]:
# your code: 3-fold cross-validation of the defaults against the study's winner on the
# training rows, then one test score for each

<details><summary>
Click here to compare your answer...
</summary>

| | 3-fold CV | folds | test |
| :-- | --: | :-- | --: |
| defaults | 0.4718 | 0.4699  0.4685  0.4771 | 0.4584 |
| **the study's winner** | **0.4512** | 0.4554  0.4421  0.4560 | **0.4341** |

**It survives.** The tuned settings win on cross-validation, **in all three folds**, and on
the test districts: 0.4341 against 0.4584, which is $2,430 of median house value.

Note the test score, 0.4341, is better than the validation score the search reported,
0.4618. Nothing is wrong: the final model was refit on all 16,512 training districts, where
every trial in the study saw only 13,209. The search was choosing settings, not building
the model it ships.

</details>

In [ ]:
# notebook 04 reads this model rather than fitting one, so it needs no training at all.
# Refit here from the study's own best_params, which does not depend on the exercise above
winner = XGBRegressor(n_estimators=ROUNDS, random_state=RSEED, n_jobs=N_JOBS,
                      **pruned_study.best_params)
winner.fit(X_train, y_train)

Path("models").mkdir(exist_ok=True)
# XGBoost's own JSON format, not pickle: it survives a library upgrade, and loading it
# cannot execute code the way unpickling can. 400 deep trees come to about 8 MB of it
winner.save_model("models/xgboost_tuned.json")

print(f"saved models/xgboost_tuned.json   test RMSE {rmse(y_test, winner.predict(X_test)):.4f}")

That file is what `04_interpreting_the_model.ipynb` opens. It is committed to the
repository as well, so that notebook runs whether or not this one has been through.

## Summary

- A **Bayesian** search models which settings have been working and draws its next trial
  from there. Forty TPE trials took the validation RMSE from the defaults' 0.4934 to
  **0.4587**, and visibly concentrated: the spread between trials fell from 0.024 to 0.004.
  Only 47% of the gain had arrived by trial 12.
- `learning_rate` and `reg_lambda` were searched on a **log scale**, because what matters
  for a rate is the ratio between values, not the difference.
- **Pruning** reports a score after every boosting round and stops a trial that is behind
  the median. **19 of the same forty trials were stopped early**, cutting roughly 40% off
  the search.
- Parameter importances are computed over the trials that **finished**, and pruning decides
  which those are. Here it removed the fast-converging high-rate trials, and
  `learning_rate`'s importance collapsed to 0.02 as a result.
- The search's own best score is optimistic, because it chose against that slice. Confirmed
  on folds and scored once, the winner reached **0.4341 against the defaults' 0.4584**.

## References & Further Reading

- [**Akiba et al., Optuna: A Next-generation Hyperparameter Optimization Framework (2019)**](https://doi.org/10.1145/3292500.3330701): The define-by-run API of section 3 and the pruning strategy of section 5.
- [**Optuna: Efficient Optimization Algorithms**](https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/003_efficient_optimization_algorithms.html): `trial.report()`, `trial.should_prune()`, and the pruners other than `MedianPruner`.
- [**Bergstra et al., Algorithms for Hyper-Parameter Optimization (2011)**](https://papers.nips.cc/paper/2011/hash/86e8f7ab32cfd12577bc2619bc635690-Abstract.html): The Tree-structured Parzen Estimator itself, and why it models the settings given the score rather than the other way round.
- [**XGBoost: Parameters**](https://xgboost.readthedocs.io/en/stable/parameter.html): All five searched here, with the ranges the documentation recommends.
- [**XGBoost: Callbacks**](https://xgboost.readthedocs.io/en/stable/python/callbacks.html): `TrainingCallback` and `after_iteration`, which section 5 subclasses.